# 03. Self-evolving OPD 루프 시뮬레이션

목표: actor와 analyzer가 같은 policy snapshot에서 함께 진화하는 SEED 구조를 장난감 환경으로 구현한다.

실행 방법:
1. 이 노트북을 위에서 아래로 실행한다.
2. iteration마다 policy weight와 success rate가 어떻게 바뀌는지 확인한다.
3. 이 노트북은 Python 표준 라이브러리만 사용한다.

실제 SEED는 LLM, rollout server, GRPO, tensor 연산, distributed training이 필요하다. 여기서는 구조만 작게 모사한다.

In [ ]:
import math
import random
from dataclasses import dataclass, field

random.seed(7)


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


def softmax(logits):
    max_logit = max(logits.values())
    exps = {key: math.exp(value - max_logit) for key, value in logits.items()}
    total = sum(exps.values())
    return {key: value / total for key, value in exps.items()}


def sample_from_probs(probs):
    threshold = random.random()
    cumulative = 0.0
    for action, prob in probs.items():
        cumulative += prob
        if threshold <= cumulative:
            return action
    return next(reversed(probs))

## 1. 장난감 policy와 환경

환경의 성공 전략은 `search -> refine -> cite` 순서를 따르는 것이다. policy는 step별 action logit을 가지고 있고, self-evolving loop가 진행되며 각 step에 맞는 좋은 행동의 weight가 커진다.

In [ ]:
ACTIONS = ["guess", "search", "refine", "cite"]
TARGET_BY_STEP = ["search", "refine", "cite"]


@dataclass
class ToyPolicy:
    logits: dict[tuple[int, str], float] = field(default_factory=lambda: {
        (step, action): (0.4 if action == "guess" else 0.0)
        for step in range(3)
        for action in ACTIONS
    })

    def action_probs(self, step, skill=None):
        adjusted = {action: self.logits[(step, action)] for action in ACTIONS}
        if skill == "gather_evidence":
            # skill-augmented context는 현재 step에 맞는 evidence gathering action을 더 그럴듯하게 만든다.
            # 실제 SEED에서는 LLM이 skill prompt가 붙은 context에서 token log-probability를 계산한다.
            adjusted[TARGET_BY_STEP[step]] += 0.9
            adjusted["guess"] -= 0.7
        return softmax(adjusted)

    def rollout(self, steps=3):
        actions = []
        for step in range(steps):
            actions.append(sample_from_probs(self.action_probs(step)))
        outcome = 1.0 if actions == TARGET_BY_STEP else 0.0
        return {"actions": actions, "outcome": outcome}


policy = ToyPolicy()
policy.action_probs(step=0)

## 2. analyzer 역할

SEED에서는 현재 policy checkpoint가 analyzer 역할도 한다. 여기서는 완료된 trajectory를 보고 evidence gathering skill을 반환하는 간단한 규칙으로 표현한다.

In [ ]:
def analyzer_from_policy(trajectory):
    """완료된 trajectory에서 hindsight skill을 만든다."""
    actions = trajectory["actions"]
    if trajectory["outcome"] == 1.0:
        return "gather_evidence"
    if "guess" in actions:
        return "gather_evidence"
    return "be_systematic"


sample = {"actions": ["guess", "guess", "cite"], "outcome": 0.0}
analyzer_from_policy(sample)

## 3. OPD update 구현

같은 action을 ordinary context와 skill context에서 다시 점수화한다. skill context에서 확률이 오른 action은 gate가 커지고, 그 action의 ordinary logit을 더 올린다.

In [ ]:
def opd_update(policy, trajectories, beta_opd=5.0, lr=0.20):
    updates = {(step, action): 0.0 for step in range(3) for action in ACTIONS}

    for traj in trajectories:
        skill = analyzer_from_policy(traj)
        for step, action in enumerate(traj["actions"]):
            ordinary_probs = policy.action_probs(step, skill=None)
            skill_probs = policy.action_probs(step, skill=skill)
            ordinary_logp = math.log(ordinary_probs[action])
            skill_logp = math.log(skill_probs[action])
            delta = skill_logp - ordinary_logp
            gate = sigmoid(beta_opd * delta)
            updates[(step, action)] += gate

    # 평균 update 크기를 제한한다. 실제 학습에서는 optimizer와 gradient clipping이 이 역할을 한다.
    normalizer = max(1, len(trajectories) * 3)
    for key, value in updates.items():
        policy.logits[key] += lr * value / normalizer


def rl_update(policy, trajectories, lr=0.15):
    """outcome-based RL을 아주 단순하게 모사한다.

    성공 trajectory의 action은 강화하고 실패 trajectory의 guess는 약하게 낮춘다.
    실제 SEED는 GRPO objective와 KL regularization을 사용한다.
    """
    for traj in trajectories:
        if traj["outcome"] > 0:
            for step, action in enumerate(traj["actions"]):
                policy.logits[(step, action)] += lr / len(traj["actions"])
        else:
            for step, action in enumerate(traj["actions"]):
                if action == "guess":
                    policy.logits[(step, "guess")] -= lr * 0.5

## 4. self-evolving loop 실행

각 iteration에서 현재 policy가 trajectory를 수집하고, 같은 policy 기준 analyzer가 skill을 만들며, update 후 policy가 다음 snapshot이 된다.

In [ ]:
def evaluate(policy, trials=300):
    successes = 0
    for _ in range(trials):
        successes += policy.rollout()["outcome"]
    return successes / trials


policy = ToyPolicy()
history = []

for iteration in range(1, 16):
    # 현재 policy snapshot이 on-policy trajectories를 만든다.
    batch = [policy.rollout() for _ in range(48)]

    # RL은 final outcome을 보고 전체 행동 분포를 조정한다.
    rl_update(policy, batch)

    # OPD는 hindsight skill이 지지하는 sampled action을 token-level로 더 조밀하게 강화한다.
    opd_update(policy, batch)

    success_rate = evaluate(policy)
    history.append((iteration, success_rate, dict(policy.logits)))
    print(f"iter={iteration:02d} success={success_rate:.3f} logits={policy.logits}")

## 5. 결과 읽기

이 장난감 환경에서는 좋은 action의 logit이 점점 커지는 방향을 볼 수 있다. 실제 SEED의 포인트는 OPD update가 그냥 성공 trajectory만 강화하는 것이 아니라, completed trajectory에서 추출된 skill이 지지한 token을 더 조밀하게 강화한다는 점이다.

In [ ]:
first = history[0]
last = history[-1]
print("initial measured success:", round(first[1], 3))
print("final measured success:", round(last[1], 3))
print("final action probabilities:")
for step in range(3):
    print(f"step {step + 1}:")
    for action, prob in policy.action_probs(step).items():
        print(f"  {action:6s}: {prob:.3f}")

## 6. 이 실습과 실제 SEED의 차이

- 실제 SEED의 action은 문자열 하나가 아니라 LLM token sequence다.
- 실제 analyzer는 completed trajectory를 읽고 자연어 hindsight skill을 생성한다.
- 실제 OPD는 ordinary branch와 skill branch의 token log-probability shift를 tensor로 계산한다.
- 실제 RL term은 GRPO, clipping, KL regularization, rollout group을 포함한다.
- 실제 학습은 distributed GPU training과 benchmark environment가 필요하다.

## 정리

- self-evolving은 최신 policy가 actor와 analyzer 역할을 모두 수행한다는 뜻이다.
- update 후 policy가 좋아지면 다음 iteration의 trajectory와 hindsight skill도 함께 바뀐다.
- OPD는 skill prompt를 inference에 남기지 않고, 학습 중 ordinary policy에 효과를 흡수한다.
- 이 구조가 SEED가 static skill prompting이나 outcome-only RL과 구분되는 핵심이다.